* 지금 insert 어떻게할지 고민했었음
* db내의 row를 뽑고, 새로운 parquet랑 비교를 해서 drop duplicate로 남은 행만 insert를 할까 했는데, data type도 다르고 불편함..
* 그래서 그냥 둘의 행수가 다르면 db에 있는 걸 지우고, 새로 insert하는걸로 하려고함~~
* 250806 이렇게 하려고 했는데, 그냥 행수 비교해서 db 행의 마지막 행 부터 loc로 인덱싱 해서 insert하려고~

In [2]:
import sys
# sys.path에 상위 폴더(..)를 추가합니다.
sys.path.append("..")

import warnings
warnings.filterwarnings('ignore')

### Scan MINIO
* TOS에서 데이터 다운로드 시작 한 시간 뒤에 MINIO에 업데이트된 파일을 분류


In [70]:
from src.voltaflow.connectors.minio import MinioConnector
import io
import pandas as pd

minio_connector = MinioConnector()

Minio connection established successfully


In [71]:
rows_to_insert = []
res = minio_connector.client.list_objects_v2(Bucket="tos-stepend")

for row in res['Contents']:
    if True: #<-TOS에서 데이터 다운로드 시작을 시작한 시간 뒤에 last modeified가 있으면서, required_downlod 컬럼이 True일 때! logic 추가 필요. 우선은 그냥 모든 파일을 가져오도록 합니다.
        rows_to_insert.append(row['Key'])

In [72]:
idx = -2
rows_to_insert[idx]

'UDKTF10115/EXP_250328_00758/07004515_20250327_JF2S_CV_ACC_StepCD_55_1_1_12CP_SOC0~100_UDKTF10115.parquet'

In [73]:
res = minio_connector.client.get_object(Bucket="tos-stepend", Key=rows_to_insert[idx])
df = pd.read_parquet(io.BytesIO(res['Body'].read()))

In [ ]:
cell_id = rows_to_insert[idx].split("/")[0]
exp_id = rows_to_insert[idx].split("/")[1]
df["Cell_ID"] = cell_id
df["EXP_ID"] = exp_id

In [75]:
df.columns = [col.lower() for col in df.columns]

### insert_to_db

* insert

In [200]:
from src.voltaflow.connectors.db import DbConnector
import os

host = os.getenv("DB_HOST")
database = "mart_db"
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")

db_conn = DbConnector(host, database, user, password, port)
db_conn.connect()
DESTINATION_TABLE_NAME = "exp_fact_tb"
query = f"SELECT * FROM {DESTINATION_TABLE_NAME} WHERE cell_id = %s and exp_id = %s;"
temp_db = db_conn.fetch_data(query, params=(cell_id, exp_id), as_dataframe=True)
temp_db = temp_db.drop("serial_id", axis=1, errors='ignore')
cols = temp_db.columns
insert_cols = ', '.join([f'{col}' for col in cols])
placeholders = ', '.join(['%s'] * len(cols))
query = f"INSERT INTO exp_fact_tb ({insert_cols}) VALUES ({placeholders})"
row_idx = temp_db.shape[0]
target_data = df[temp_db.columns].loc[row_idx:, cols].values
db_conn.cursor.executemany(query, target_data)
db_conn.commit()

데이터베이스 'mart_db'에 성공적으로 연결되었습니다.
트랜잭션이 커밋되었습니다.


In [11]:
from src.voltaflow.connectors.db import DbConnector
import os

host = os.getenv("DB_HOST")
database = "mart_db"
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")


cell_id = "DA05B19225"
exp_id = "EXP_241113_00788"

db_conn = DbConnector(host, database, user, password, port)
db_conn.connect()
DESTINATION_TABLE_NAME = "exp_fact_tb"
query = f"SELECT * FROM {DESTINATION_TABLE_NAME} WHERE cell_id = %s and exp_id = %s;"
temp_db = db_conn.fetch_data(query, params=(cell_id, exp_id), as_dataframe=True)
temp_db = temp_db.drop("serial_id", axis=1, errors='ignore')

데이터베이스 'mart_db'에 성공적으로 연결되었습니다.


In [12]:
temp_db

,cell_id,exp_id,stepno,steptype,code,totalcycle,voltage,current,capacity,power,...,avgvoltage,avgcurrent,chargecapacity,dischargecapacity,chargewatthour,dischargewatthour,cvendtime,testmode,isrptcapa,isrptdcir
0,DA05B19225,EXP_241113_00788,2,Rest,Time Complete,1,3.332281,-0.002527,0.000000,0.000,...,3.3322,-0.001900,0.000000,0.000000,0.000,0.000,0.00,RPT,False,False
1,DA05B19225,EXP_241113_00788,5,Discharge,Voltage Complete,2,2.499997,-14.148884,55.919186,-35.372,...,3.2119,-14.149401,0.000000,55.919186,0.000,179.589,0.00,RPT,False,False
2,DA05B19225,EXP_241113_00788,6,Rest,Time Complete,2,2.823481,-0.000405,0.000000,0.000,...,2.7848,-0.002800,0.000000,0.000000,0.000,0.000,0.00,RPT,False,False
3,DA05B19225,EXP_241113_00788,7,Charge,Current Complete,2,3.649631,2.829780,56.659817,10.327,...,3.3489,14.071700,56.659817,0.000000,189.637,0.000,137.75,RPT,True,False
4,DA05B19225,EXP_241113_00788,8,Rest,Time Complete,2,3.468288,-0.003028,0.000000,0.000,...,3.4878,-0.002800,0.000000,0.000000,0.000,0.000,0.00,RPT,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1268,DA05B19225,EXP_241113_00788,28,Rest,Time Complete,317,3.330079,0.000000,0.000000,0.000,...,0.0000,0.000000,0.000000,0.000000,0.000,0.000,0.00,In-situ,False,False
1269,DA05B19225,EXP_241113_00788,25,Charge,Voltage Complete,318,3.650031,24.971142,0.633348,91.145,...,3.4512,26.412800,0.633348,0.000000,2.184,0.000,0.00,In-situ,False,False
1270,DA05B19225,EXP_241113_00788,26,Rest,Time Complete,318,3.432760,0.000000,0.000000,0.000,...,0.0000,0.000000,0.000000,0.000000,0.000,0.000,0.00,In-situ,False,False
1271,DA05B19225,EXP_241113_00788,27,Discharge,Voltage Complete,318,3.264928,-27.907520,0.633962,-91.116,...,3.3017,-27.597400,0.000000,0.633962,0.000,2.093,0.00,In-situ,False,False
